# Lab 2.1 · Data Profiling และทำความสะอาดข้อมูลด้วย AI 

**คาบ 2 · เสาร์ 27 ก.ย. 2026 · 45 นาที**

ข้อมูล `sales_raw.csv` คือยอดขายชุดเดียวกับที่ใช้ใน Lab 1 แต่ดึงมาจากระบบ POS โดยตรง จึงมีปัญหาปนมา เป้าหมายคือได้ `sales_clean.csv` ที่ถูกต้อง แล้วนำไปใช้ต่อใน Dashboard ของ Lab 2.2

| ขั้น | เวลา | ทำอะไร |
|---|---|---|
| 1 | 5 นาที | โหลดและดูข้อมูลครั้งแรก |
| 2 | 10 นาที | Profiling: ให้ AI หาปัญหา **แต่ยังไม่แก้** แล้วตรวจคำตอบ |
| 3 | 5 นาที | ตัดสินใจว่าจะจัดการแต่ละปัญหาอย่างไร |
| 4 | 20 นาที | ทำความสะอาดด้วย AI |
| 5–6 | 5 นาที | Verify และ export |

**วิธีใช้กับ AI:** ใช้ Gemini ที่อยู่ใน Colab หรือคัดลอก prompt ไปใช้กับ Claude / ChatGPT แล้ววางโค้ดกลับมา

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) เซลล์นี้มีฟังก์ชัน `check()` สำหรับตรวจคำตอบโดยไม่เฉลยตัวเลข และ `validate()` สำหรับตรวจข้อมูลที่ทำความสะอาดแล้ว

In [ ]:
# ===== เซลล์ตัวช่วยของ Lab 2.1 · รันก่อน ไม่ต้องแก้ =====
import base64, json, os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

_EXP = json.loads(base64.b64decode(
    "eyJyYXdfcm93cyI6IDUzMzU3LCAiZHVwX3Jvd3MiOiAyNjUsICJiYWRfZGF0ZXRpbWUiOiAxMTYyLCAiYmFkX2JyYW5jaCI6IDEwNzEsICJwcmljZV9ub3RfbnVtYmVyIjogNjEsICJwcmljZV9uZWdhdGl2ZSI6IDI1LCAicXR5X3plcm8iOiAyMCwgIm1pc3NpbmdfcHJvZHVjdCI6IDE1fQ=="
).decode())
_LABEL = {
    "raw_rows": "จำนวนแถวทั้งหมด",
    "dup_rows": "แถวที่ซ้ำกันทุกคอลัมน์ (ไม่นับแถวแรก)",
    "bad_datetime": "datetime ที่ไม่ใช่รูปแบบ ค.ศ. มาตรฐาน (YYYY-MM-DDT…)",
    "bad_branch": "ชื่อสาขาที่ไม่ใช่ 5 ชื่อมาตรฐาน",
    "price_not_number": "unit_price ที่แปลงเป็นตัวเลขไม่ได้",
    "price_negative": "unit_price ที่ติดลบ",
    "qty_zero": "qty เท่ากับ 0",
    "missing_product": "product_id ว่าง",
}
_HINT = {
    "dup_rows": "ใช้ df.duplicated() ทั้งแถว ไม่ใช่ duplicated('order_id') เพราะ 1 บิลมีหลายแถว",
    "bad_datetime": "นับทั้งปี พ.ศ. (25xx-…) และรูปแบบ DD/MM/YYYY รวมกัน",
    "bad_branch": "ช่องว่างท้ายชื่อก็นับว่าไม่มาตรฐาน ลองดู df['branch'].value_counts()",
    "price_not_number": "ลอง pd.to_numeric(df['unit_price'], errors='coerce').isna()",
    "price_negative": "ต้องแปลงเป็นตัวเลขก่อนจึงเทียบ < 0 ได้",
    "missing_product": "อ่านไฟล์ด้วย keep_default_na=False ค่าว่างจึงเป็น '' ไม่ใช่ NaN",
}
STD_BRANCHES = ["สยาม", "สีลม", "อารีย์", "บางนา", "มหาวิทยาลัย"]


def check(name, value):
    """เทียบคำตอบกับเฉลยโดยไม่บอกตัวเลข"""
    exp = _EXP[name]
    label = _LABEL[name]
    if value is None:
        print(f"⬜ {label}: ยังไม่ได้ใส่ค่า")
    elif int(value) == exp:
        print(f"✅ {label}: {int(value):,}")
    elif abs(int(value) - exp) <= max(2, exp * 0.03):
        print(f"🟡 {label}: {int(value):,} ใกล้แล้ว แต่ยังไม่ตรง · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")
    else:
        print(f"❌ {label}: {int(value):,} ยังไม่ถูก · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")


def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_csv(name, dtype=str, keep_default_na=False)


def validate(clean, reference="sales.csv"):
    """ตรวจ sales_clean ว่าพร้อมใช้ใน Dashboard และเทียบกับไฟล์อ้างอิง"""
    ok = True
    def rep(passed, msg):
        nonlocal ok
        ok &= bool(passed)
        print(("✅ " if passed else "❌ ") + msg)

    cols = ["order_id", "datetime", "branch", "product_id", "qty", "unit_price",
            "customer_id", "payment_method", "channel"]
    rep(list(clean.columns) == cols, "คอลัมน์ครบและเรียงเหมือน sales.csv")
    dt = clean["datetime"].astype(str)
    bad_dt = ~dt.str.match(r"^20(25|26)-\d\d-\d\dT\d\d:\d\d:\d\d\+07:00$")
    rep(bad_dt.sum() == 0, f"datetime เป็น YYYY-MM-DDTHH:MM:SS+07:00 ทุกแถว (ผิด {bad_dt.sum():,} แถว)")
    bad_b = ~clean["branch"].isin(STD_BRANCHES)
    rep(bad_b.sum() == 0, f"ชื่อสาขาเหลือ 5 ชื่อมาตรฐาน (ผิด {bad_b.sum():,} แถว)")
    price = pd.to_numeric(clean["unit_price"], errors="coerce")
    rep(price.notna().all() and (price > 0).all(), "unit_price เป็นตัวเลขมากกว่า 0 ทุกแถว")
    qty = pd.to_numeric(clean["qty"], errors="coerce")
    rep(qty.notna().all() and (qty > 0).all(), "qty เป็นตัวเลขมากกว่า 0 ทุกแถว")
    rep((clean["product_id"].astype(str).str.match(r"^P0[0-4]\d$")).all(), "product_id ไม่ว่าง และอยู่ในช่วง P001–P040")
    rep(clean.duplicated().sum() == 0, "ไม่มีแถวซ้ำ")

    if os.path.exists(reference):
        ref = pd.read_csv(reference, dtype=str, keep_default_na=False)
        key = ["order_id", "product_id"]
        m = ref.merge(clean.astype(str), on=key, how="outer", suffixes=("_ref", ""), indicator=True)
        only_ref = (m["_merge"] == "left_only").sum()
        only_clean = (m["_merge"] == "right_only").sum()
        both = m[m["_merge"] == "both"]
        diff_min = (both["datetime_ref"].str[:16] != both["datetime"].str[:16]).sum()
        diff_branch = (both["branch_ref"] != both["branch"]).sum()
        rev = lambda d: (pd.to_numeric(d["qty"]) * pd.to_numeric(d["unit_price"])).sum()
        print("\n— เทียบกับไฟล์อ้างอิง sales.csv —")
        print(f"แถวที่คุณตัดทิ้ง: {only_ref:,} · แถวเกิน: {only_clean:,}")
        print(f"วันเวลาไม่ตรง (ระดับนาที): {diff_min:,} · สาขาไม่ตรง: {diff_branch:,}")
        print(f"ยอดขายรวม: ของคุณ ฿{rev(clean):,.0f} · อ้างอิง ฿{rev(ref):,.0f} · ต่าง ฿{rev(clean) - rev(ref):,.0f}")
        rep(only_clean == 0 and diff_min == 0 and diff_branch == 0, "ทุกแถวที่เก็บไว้ตรงกับข้อมูลอ้างอิง")
        rep(only_ref <= 100, f"ตัดทิ้งไม่เกิน 100 แถว (ตัดไป {only_ref:,}) · ถ้าเกิน ตรวจว่าไม่ได้ลบแถวซ้ำด้วย order_id หรือทิ้งแถวที่แก้ได้")
    print("\n🎉 พร้อม export" if ok else "\nยังมีจุดต้องแก้ ลองส่งข้อความ ❌ ให้ AI ช่วยหาสาเหตุ")
    return ok

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก (5 นาที)

อัปโหลด `sales_raw.csv` และ `sales.csv` จากโฟลเดอร์ `data/` (ถ้าใช้ Colab จะมีปุ่มให้อัปโหลด)

เราอ่าน **ทุกคอลัมน์เป็นข้อความ** โดยตั้งใจ เพราะถ้าให้ pandas เดาชนิดข้อมูลเอง ปัญหาบางอย่างจะถูกซ่อน เช่น ค่าว่างกลายเป็น NaN หรือตัวเลขที่ปนข้อความทำให้ทั้งคอลัมน์กลายเป็นข้อความโดยไม่บอกเรา

In [ ]:
df = load_csv("sales_raw.csv")
load_csv("sales.csv")  # ไฟล์อ้างอิงสำหรับ validate() ในขั้นที่ 5
print(f"{len(df):,} แถว × {df.shape[1]} คอลัมน์")
df.head(8)

**คำถามชวนคิด:** ข้อมูลนี้มีกี่แถวมากกว่า `sales.csv` ของ Lab 1 (53,092 แถว) และคุณเดาว่าแถวที่เกินมาจากอะไร

## ขั้นที่ 2 · Profiling: หาปัญหาก่อน ยังไม่แก้ (10 นาที)

**Prompt**
```
ช่วยทำ data profiling ของ DataFrame df ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- จำนวนค่าว่างและค่าไม่ซ้ำของแต่ละคอลัมน์
- จำนวนแถวที่ซ้ำกันทุกคอลัมน์
- จำนวน datetime ที่ไม่ใช่รูปแบบ YYYY-MM-DDT... ปี ค.ศ. พร้อมตัวอย่าง
- ชื่อสาขาทั้งหมดพร้อมจำนวนแถว
- จำนวน unit_price ที่แปลงเป็นตัวเลขไม่ได้ พร้อมตัวอย่าง และจำนวนที่ติดลบ
- จำนวนแถว qty = 0 และ product_id ว่าง
เก็บจำนวนแต่ละข้อไว้ในตัวแปร n_dup, n_bad_dt, n_bad_branch, n_price_text, n_price_neg, n_qty_zero, n_missing_product
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```
ทำไมต้อง “รายงานก่อน แก้ทีหลัง”: ถ้าสั่งแก้ทันที AI อาจลบหรือเปลี่ยนข้อมูลเงียบ ๆ โดยเราไม่รู้ว่าเปลี่ยนอะไรไปบ้าง

In [ ]:
# ===== Data profiling: รายงานอย่างเดียว ไม่แก้ df =====
print("=== ค่าว่างและค่าไม่ซ้ำของแต่ละคอลัมน์ ===")
print(pd.DataFrame({
    "ค่าว่าง": (df == "").sum(),
    "ค่าไม่ซ้ำ": df.nunique(),
}))

# 1) แถวที่ซ้ำกันทุกคอลัมน์ (ไม่นับแถวแรก)
n_dup = int(df.duplicated().sum())
print(f"\n=== แถวซ้ำทุกคอลัมน์: {n_dup:,} ===")

# 2) datetime ที่ไม่ใช่ ISO ปี ค.ศ. (20xx-MM-DDT...)
dt = df["datetime"].str.strip()
iso_ce = dt.str.match(r"^20\d\d-\d\d-\d\dT")
n_bad_dt = int((~iso_ce).sum())
print(f"\n=== datetime ไม่มาตรฐาน: {n_bad_dt:,} ===")
kind = pd.Series("อื่น ๆ", index=dt.index)
kind[dt.str.match(r"^25\d\d-\d\d-\d\dT")] = "ISO ปี พ.ศ."
kind[dt.str.match(r"^\d{1,2}/\d{1,2}/20\d\d")] = "DD/MM/YYYY ค.ศ."
kind[dt.str.match(r"^\d{1,2}/\d{1,2}/25\d\d")] = "DD/MM/YYYY พ.ศ."
bad_kind = kind[~iso_ce]
print(bad_kind.value_counts().to_string())
for k in bad_kind.unique():
    print(f"  ตัวอย่าง {k}: {dt[bad_kind[bad_kind == k].index[:3]].tolist()}")

# 3) ชื่อสาขา
print("\n=== ชื่อสาขาทั้งหมด (repr เพื่อให้เห็นช่องว่าง) ===")
for name, n in df["branch"].value_counts().items():
    print(f"  {name!r:>20}: {n:,}")
n_bad_branch = int((~df["branch"].isin(STD_BRANCHES)).sum())
print(f"ชื่อสาขาไม่มาตรฐาน: {n_bad_branch:,}")

# 4) unit_price
price = pd.to_numeric(df["unit_price"], errors="coerce")
n_price_text = int(price.isna().sum())
n_price_neg = int((price < 0).sum())
print(f"\n=== unit_price แปลงเป็นตัวเลขไม่ได้: {n_price_text:,} ===")
print(f"  ตัวอย่าง: {df.loc[price.isna(), 'unit_price'].unique()[:8].tolist()}")
print(f"=== unit_price ติดลบ: {n_price_neg:,} ===")
print(f"  ตัวอย่าง: {df.loc[price < 0, 'unit_price'].unique()[:8].tolist()}")

# 5) qty = 0 และ product_id ว่าง
qty = pd.to_numeric(df["qty"], errors="coerce")
n_qty_zero = int((qty == 0).sum())
n_missing_product = int((df["product_id"].str.strip() == "").sum())
print(f"\n=== qty = 0: {n_qty_zero:,} · qty แปลงไม่ได้: {int(qty.isna().sum()):,} ===")
print(f"=== product_id ว่าง: {n_missing_product:,} ===")

### ✔️ ตรวจคำตอบ
ได้ ✅ ครบ 8 ข้อแล้วค่อยไปขั้นต่อไป ถ้าได้ 🟡 หรือ ❌ ให้อ่านคำใบ้แล้วสั่ง AI แก้โค้ด profiling

In [ ]:
# ตัวแปร n_* ถูกคำนวณในเซลล์ profiling ด้านบนแล้ว รันเซลล์นี้เพื่อตรวจ

check("raw_rows", len(df))
check("dup_rows", n_dup)
check("bad_datetime", n_bad_dt)
check("bad_branch", n_bad_branch)
check("price_not_number", n_price_text)
check("price_negative", n_price_neg)
check("qty_zero", n_qty_zero)
check("missing_product", n_missing_product)

## ขั้นที่ 3 · ตัดสินใจก่อนลงมือ (5 นาที · คุยกับเพื่อนข้าง ๆ)

AI เสนอวิธีแก้ได้ แต่ **คนที่รู้ธุรกิจต้องเป็นคนตัดสิน** ดับเบิลคลิกเซลล์นี้เพื่อกรอกตาราง

| ปัญหา | ทางเลือก | ตัดสินใจ | เหตุผล |
|---|---|---|---|
| แถวซ้ำทุกคอลัมน์ | ลบ / เก็บ | ลบ | POS ส่งแถวเดิมซ้ำ ถ้าเก็บไว้จะนับยอดขายเกิน |
| วันที่ พ.ศ. และ DD/MM/YYYY | แปลง / ลบ | แปลง | เป็นแค่รูปแบบที่ต่างกัน ข้อมูลถูกต้อง แปลงได้ครบทุกแถว |
| ชื่อสาขาสะกดต่าง | map เป็นชื่อมาตรฐาน / ลบ | map | รู้ว่าแต่ละชื่อคือสาขาไหน (ทีม POS ยืนยันว่า "ม." คือมหาวิทยาลัย) |
| ราคามีคำว่า “บาท” | ตัดคำแล้วแปลง / ลบ | ตัดคำแล้วแปลง | ตัวเลขยังถูกต้อง แค่มีหน่วยติดมา |
| ราคาติดลบ | ลบ / แปลงเป็นบวก / เก็บไว้เป็นการคืนเงิน | แปลงเป็นบวก | ไม่มีการคืนเงินในระบบ จึงเป็นเครื่องหมายลบที่ใส่ผิด ตัวเลขตรงกับราคาจริงของสินค้า |
| qty = 0 | ลบ / เก็บ | ลบ | เป็นบิลที่ยกเลิก ไม่ใช่ยอดขาย |
| product_id ว่าง | ลบ / เดาจากราคา | ลบ | หลายสินค้าราคาเท่ากัน เดาไม่ได้แน่นอน และมีแค่ 15 แถว |

> ข้อมูลเพิ่มเติมจากร้าน (ผู้สอนเฉลยเมื่อมีคนถาม): ฝ่ายบัญชียืนยันว่า **ไม่มีการคืนเงินในระบบ POS นี้** · ทีม POS ยืนยันว่า **“ม.” คือสาขามหาวิทยาลัย** · qty = 0 คือ **บิลที่ยกเลิก**

## ขั้นที่ 4 · ทำความสะอาดด้วย AI (20 นาที)

**Prompt ตั้งต้น** (แก้ส่วนการตัดสินใจให้ตรงกับตารางของคุณ)

```
เขียนโค้ด pandas ทำความสะอาด df (อ่านมาเป็น str ทุกคอลัมน์ ค่าว่างเป็น "")
- ทำงานบน clean = df.copy() ห้ามแก้ df
- แยกแต่ละปัญหาเป็นขั้นตอนชัดเจน และเก็บ log เป็นรายการ (ขั้นตอน, จำนวนแถวที่กระทบ, การตัดสินใจ)
- ลบแถวที่ซ้ำกันทุกคอลัมน์ (ห้ามใช้ order_id อย่างเดียว เพราะ 1 บิลมีหลายแถว)
- แปลง datetime ทุกแถวเป็นรูปแบบ YYYY-MM-DDTHH:MM:SS+07:00 ปี ค.ศ.
  ต้องรองรับ ISO ปี ค.ศ., ISO ปี พ.ศ. และ DD/MM/YYYY HH:MM (ซึ่งอาจเป็นปี พ.ศ. ด้วย) ถ้าไม่มีวินาทีให้ใส่ 00
- ตัดช่องว่างชื่อสาขา แล้ว map ให้เหลือ: สยาม, สีลม, อารีย์, บางนา, มหาวิทยาลัย
- unit_price: ตัดคำว่า "บาท" แปลงเป็นตัวเลข ราคาติดลบให้ [ตามที่ตัดสินใจ]
- ลบแถว qty = 0 และแถว product_id ว่าง
- ผลลัพธ์ต้องมีคอลัมน์และลำดับเหมือนเดิม unit_price เป็นจำนวนเต็ม
- สุดท้ายแสดง log เป็นตาราง และจำนวนแถวก่อน/หลัง
```

**กับดักที่ต้องระวัง** ถ้า `validate()` ขึ้น ❌ ให้ลองดูสามจุดนี้ก่อน
1. มีวันที่ **9 แถว** ที่เป็นรูปแบบ DD/MM/YYYY **และ** ปี พ.ศ. พร้อมกัน
2. AI ชอบใช้ `pd.to_datetime()` ซึ่งอาจแปลง 01/04 เป็น 4 ม.ค. (สลับวันกับเดือน) และแปลงเป็นเวลา UTC
3. ถ้าแก้ชื่อสาขาด้วย map โดยไม่ strip ก่อน “สยาม␣” จะไม่ถูกแก้

In [ ]:
# ===== ทำความสะอาดข้อมูล: ทำงานบน clean ไม่แตะ df =====
clean = df.copy()
log = []

def add_log(step, n, decision):
    log.append({"ขั้นตอน": step, "แถวที่กระทบ": int(n), "การตัดสินใจ": decision})

# 1) ลบแถวที่ซ้ำกันทุกคอลัมน์ (ไม่ใช้ order_id อย่างเดียว เพราะ 1 บิลมีหลายแถว)
dup = clean.duplicated()
clean = clean[~dup]
add_log("ลบแถวซ้ำทุกคอลัมน์", dup.sum(), "ลบ · เป็นแถวที่ POS ส่งซ้ำ")

# 2) แปลง datetime ให้เป็น YYYY-MM-DDTHH:MM:SS+07:00 ปี ค.ศ.
#    แยกส่วนด้วย regex เอง ไม่ใช้ pd.to_datetime() เพราะอาจสลับวัน/เดือนและแปลงเป็น UTC
ISO = r"^(?P<y>\d{4})-(?P<m>\d\d)-(?P<d>\d\d)T(?P<H>\d\d):(?P<M>\d\d)(?::(?P<S>\d\d))?"
DMY = r"^(?P<d>\d{1,2})/(?P<m>\d{1,2})/(?P<y>\d{4})\s+(?P<H>\d{1,2}):(?P<M>\d\d)(?::(?P<S>\d\d))?"
dt = clean["datetime"].str.strip()
parts = dt.str.extract(ISO)
is_dmy = parts["y"].isna()
parts[is_dmy] = dt[is_dmy].str.extract(DMY)[parts.columns]
assert parts["y"].notna().all(), f"มี datetime ที่อ่านไม่ได้: {dt[parts['y'].isna()].head().tolist()}"

year = parts["y"].astype(int)
is_be = year >= 2500
year = year.where(~is_be, year - 543)
parts["S"] = parts["S"].fillna("00")
new_dt = (year.astype(str) + "-" + parts["m"].str.zfill(2) + "-" + parts["d"].str.zfill(2)
          + "T" + parts["H"].str.zfill(2) + ":" + parts["M"] + ":" + parts["S"] + "+07:00")
n_changed = (new_dt != clean["datetime"]).sum()
clean["datetime"] = new_dt
add_log("แปลง datetime เป็น ISO ปี ค.ศ.", n_changed,
        f"แปลง · พ.ศ.→ค.ศ. {is_be.sum():,} แถว, DD/MM/YYYY {is_dmy.sum():,} แถว (ทั้งสองแบบ {(is_be & is_dmy).sum():,})")

# 3) ชื่อสาขา: strip ก่อน แล้ว map เป็นชื่อมาตรฐาน
BRANCH_MAP = {
    "Siam": "สยาม", "สาขาสยาม": "สยาม",
    "Silom": "สีลม",
    "Ari": "อารีย์", "อารีย": "อารีย์",
    "Bangna": "บางนา",
    "มหาลัย": "มหาวิทยาลัย", "ม.": "มหาวิทยาลัย",  # ทีม POS ยืนยันว่า "ม." คือมหาวิทยาลัย
}
branch = clean["branch"].str.strip()
branch = branch.replace(BRANCH_MAP)
unknown = ~branch.isin(STD_BRANCHES)
assert not unknown.any(), f"มีชื่อสาขาที่ยังไม่รู้จัก: {branch[unknown].unique().tolist()}"
add_log("map ชื่อสาขาเป็น 5 ชื่อมาตรฐาน", (branch != clean["branch"]).sum(), "strip ช่องว่าง แล้ว map ชื่อสะกดต่าง")
clean["branch"] = branch

# 4) unit_price: ตัดคำว่า "บาท" แล้วแปลงเป็นตัวเลข
price_str = clean["unit_price"].str.replace("บาท", "", regex=False).str.strip()
has_text = price_str != clean["unit_price"]
price = pd.to_numeric(price_str, errors="raise")
add_log("ตัดคำว่า 'บาท' ใน unit_price", has_text.sum(), "ตัดคำแล้วแปลงเป็นตัวเลข")

# ราคาติดลบ: ร้านไม่มีการคืนเงิน จึงถือว่าใส่เครื่องหมายลบผิด แล้วแปลงเป็นค่าบวก
neg = price < 0
price = price.abs()
add_log("แปลง unit_price ติดลบเป็นบวก", neg.sum(), "แปลงเป็นบวก · ไม่มีการคืนเงินในระบบ POS")
clean["unit_price"] = price.round().astype(int).astype(str)

# 5) ลบบิลยกเลิก (qty = 0) และแถวที่ไม่มี product_id
qty0 = pd.to_numeric(clean["qty"]) == 0
clean = clean[~qty0]
add_log("ลบแถว qty = 0", qty0.sum(), "ลบ · เป็นบิลที่ยกเลิก")

no_prod = clean["product_id"].str.strip() == ""
clean = clean[~no_prod]
add_log("ลบแถว product_id ว่าง", no_prod.sum(), "ลบ · เดาสินค้าจากราคาไม่ได้แน่นอน")

# 6) ตรวจแถวซ้ำอีกรอบหลังปรับรูปแบบ (บางแถวอาจซ้ำกันหลังแปลงวันที่/สาขา)
dup2 = clean.duplicated()
clean = clean[~dup2].reset_index(drop=True)
add_log("ลบแถวซ้ำหลังปรับรูปแบบ", dup2.sum(), "ลบ · ซ้ำกันเมื่อปรับรูปแบบแล้ว")

clean = clean[df.columns]
cleaning_log = pd.DataFrame(log)
print(cleaning_log.to_string(index=False))
print(f"\nก่อน {len(df):,} แถว → หลัง {len(clean):,} แถว (ตัดไป {len(df) - len(clean):,})")

## ขั้นที่ 5 · Verify (5 นาที)

`validate()` ตรวจว่าข้อมูลพร้อมใช้ใน Dashboard และเทียบกับ `sales.csv` (ข้อมูลอ้างอิงที่ใช้ใน Lab 1) ต้องได้ ✅ ทุกข้อก่อน export

ถ้ามี ❌ ให้คัดลอกข้อความทั้งหมดไปให้ AI แล้วพิมพ์ว่า “ช่วยหาสาเหตุ และอธิบายก่อนแก้”

In [ ]:
validate(clean)

## ขั้นที่ 6 · Export
ไฟล์ `sales_clean.csv` คือสิ่งที่จะนำไปใช้ต่อใน Lab 2.2 ส่วน `cleaning_log.csv` คือหลักฐานว่าเราทำอะไรกับข้อมูลไปบ้าง ให้แนบไว้ใน repo ด้วย

In [ ]:
clean.to_csv("sales_clean.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8-sig")
print("บันทึก sales_clean.csv และ cleaning_log.csv แล้ว")

try:
    from google.colab import files
    files.download("sales_clean.csv")
    files.download("cleaning_log.csv")
except ImportError:
    pass

### นำไปใช้ต่อใน Lab 2.2
1. คัดลอก `sales_clean.csv` ไปวางใน `public/` ของโปรเจกต์ Dashboard
2. เปลี่ยนชื่อไฟล์เป็น `sales.csv` (ทับไฟล์เดิม)
3. รีเฟรช Dashboard แล้วสังเกตว่า KPI เปลี่ยนไปจาก Lab 1 เท่าไร และเพราะอะไร

## ขั้นที่ 7 · สะพานไป Lab 2.2: ค่าเฉลี่ยหลอกตาได้

ข้อมูลสะอาดแล้ว ลองดูว่า “ลูกค้าจ่ายต่อบิลเท่าไร” มีคำตอบเดียวจริงไหม

In [ ]:
# ยอดต่อบิล: 1 บิล = รวมทุกแถวของ order_id เดียวกัน
bill = (pd.to_numeric(clean["qty"]) * pd.to_numeric(clean["unit_price"])).groupby(clean["order_id"]).sum()
print(f"ค่าเฉลี่ยต่อบิล  ฿{bill.mean():,.2f}")
print(f"ค่ากลางต่อบิล   ฿{bill.median():,.2f}")
print(f"90% ของบิลไม่เกิน ฿{bill.quantile(0.9):,.0f}")

import matplotlib.pyplot as plt
ax = bill.plot.hist(bins=range(0, 760, 20), color="#0F8B8D", figsize=(9, 3.5))
ax.axvline(bill.mean(), color="#B4461F", linestyle="--", label=f"mean {bill.mean():.0f}")
ax.axvline(bill.median(), color="#16233B", label=f"median {bill.median():.0f}")
ax.set_xlabel("Revenue per bill (THB)"); ax.set_ylabel("Bills"); ax.legend()
plt.show()
# หมายเหตุ: ใช้ป้ายภาษาอังกฤษ เพราะ matplotlib ใน Colab ไม่มีฟอนต์ไทย จะแสดงเป็นกล่องสี่เหลี่ยม

**คำถามชวนคิด:** ถ้าจะตั้งโปรฯ “ซื้อครบ X บาท รับส่วนลด” ควรใช้ค่าเฉลี่ยหรือค่ากลางเป็นฐาน และ X ควรเป็นเท่าไร


---
### 🚀 เสร็จเร็ว? ลองต่อ
- ให้ AI เขียนฟังก์ชัน `clean_sales(path)` ที่รวมทุกขั้น เรียกครั้งเดียวได้ไฟล์สะอาด (จะใช้ต่อในคาบ 7 ตอนทำ pipeline อัตโนมัติ)
- เพิ่มคอลัมน์ `date` และ `hour` ลงในข้อมูลสะอาด แล้วตรวจว่าไม่มีวันที่เลื่อนเพราะ timezone
- ลองอ่าน `customers.csv` แล้วทำ profiling ด้วย prompt เดิม มีปัญหาอะไรไหม